# 2a Escola de IA e Computação do CBPF (17-21/Ago/2026)
## Curso de Astrofísica Computacional
## Aula 2: Medidas de redshifts fotométricos de galáxias com aprendizado de máquina
## 2.a) Preparação dos dados

## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from astropy import units as u
from astropy.coordinates import SkyCoord

## Introdução

### Sobre os dados


Nesta atividade, usaremos um conjunto de dados previamente combinado e parcialmente tratado, composto por catálogos que foram disponibilizados publicamente pelos seus levantamentos de origem. Estes dados foram obtidos a partir de dois métodos observacionais diferentes: a fotometria e a espectroscopia. 

As grandezas físicas que vamos utilizar no exemplo de aplicação de métodos de aprendizado de máquina são as magnitudes aparentes de galáxias, suas cores e seus respectivos erros cumprindo o papel de atributos observáveis (ou _features_) que usaremos para treinar o modelo de aprendizagem de máquina, e o _redshift_ (z), no papel de alvo a ser estimado (ou _target_).

#### Magnitude aparente 

A magnitude é uma medida adimensional baseada em escala logaritmica associada à quantidade de energia luminosa que viaja a partir de uma fonte celeste e atinge o sensor do telescópio. A magnitude possui uma escala invertida ao brilho. Objetos mais brilhantes tem magnitudes menores. A magnitude absoluta quantifica o brilho intrínseco de um objeto e a magnitude aparente (a que conseguimos medir diretamente), é influenciada pela distância do objeto ao observador. 

#### Redshift 

O _redshift_ (z) é o desvio para o vermelho causado pelo efeito doppler cosmológico, que pode ser medido diretamente e utilizado para estimar distâncias de objetos extragalácticos.  

### Levantamentos fotométricos 


Os dados fotométricos são oriundos do [Dark Energy Survey (DES)](https://www.darkenergysurvey.org/), um levantamento fotométrico em 5 bandas do óptico ao infravermelho (_grizY_) que tem como principal objetivo a determinação da equação de estado da energia escura. O DES observou ~700 milhões de objetos detectados em ~5000 graus quadrados no hemisfério sul durante 6 anos. Os artigos com os principais resultados da análise dos dados dos seis primeiros anos de observação estão disponíveis [nesta página](https://www.darkenergysurvey.org/des-y6-cosmology-results-papers/).

<img align="center" src=https://www.darkenergysurvey.org/wp-content/uploads/2026/01/figintro-1024x500.png  width=750 style="padding: 20px"> <br> 
Figura: Footprint Data Release 2 (fonte: [www.darkenergysurvey.org](https://www.darkenergysurvey.org/wp-content/uploads/2026/01/figintro-1024x500.png)).  

O [segundo _data release_ (DR2)](https://des.ncsa.illinois.edu/releases/dr2), já contendo os dados dos seis anos de observação, está disponível para o público e pode ser acessado pelo [LIneA Science Server](https://scienceserver.linea.org.br) **INCLUIR DESCRIÇÃO DO LIneA e do JupyterHub.**

Os dados que vamos utilizar já foram extraídos das imagens do céu, pré-processados e disponibilizados de forma tabular no banco de dados. Os atributos observáveis (ou _features_) que usaremos para treinar o modelo de aprendizagem de máquina serão as magnitudes* aparentes nas 5 bandas _grizY_ e seus respectivos erros.


### Levantamentos espectroscópicos 


Para treinar os algoritmos de aprendizado supervisionado de máquina, precisamos fornecer medidas conhecidas (ou _labels_) da variável que estamos tentando calcular, neste caso, de _redshift_, de um subconjunto dos dados. Para isto, vamos utilizar dados obtidos de levantamentos espectroscópicos (spec-z) que, devido a sua enorme precisão quando comparada à fotometria, são considerados como valores "verdadeiros", ou seja, com erros nulos. 

As medidas de spec-z que vamos utilizar estão disponíveis em um catálogo, já associado aos dados fotométricos do DES, que foi produzido como parte da contribuição do LIneA para o projeto. A equipe de cientistas de dados do LIneA fez a curadoria das medidas de spec-z dos principais levantamentos disponíveis até o último ano de observações do DES (28 levantamentos) e montou um catálogo único de spec-zs, com dados limpos e homogeneizados. Esses dados foram associados às galáxias do DES através das posições em coordenadas equatoriais (R.A. e Dec.) através da técnica conhecida como _spatial cross-matching_. Este catálogo foi utilizado para alimentar os conjuntos de treinamento utilizados na produção de medidas de redshift fotométrico (photo-z) que estão sendo utilizadas nos artigos de cosmologia em preparação.  

<img align="center" src=https://dev.linea.org.br/~julia/specz_spatial_dist.png  width=500 style="padding: 20px"><img align="center" src=https://dev.linea.org.br/~julia/specz_matched_spatial_dist.png width=500 style="padding: 20px">  

Na figura acima observamos a distribuição espacial heterogênea resultante da combinação de várias fontes de dados espectroscópicos da literatura. No segundo mapa, vemos em destaque as medidas de spec-z que caem dentro da regição observada pelo DES (_footprint_). A caracterização completa desta amostra está disponível na [área de contribuição dos usuários](https://github.com/linea-it/jupyterhub-tutorial/blob/main/users-notebooks/spectroscopic-redshifts.ipynb), no mesmo repositório dos tutoriais do LIneA JupyterHub. 

A tabela abaixo traz a descrição das colunas que vamos utilizar neste _notebook_. 

|Coluna | Descrição |
|---|---|
|RA | Abreviação de _Right Ascension_, coordenada celeste equatorial no sistema J2000 (unidade: graus)|
|DEC | Abreviação de _Declination_, coordenada celeste equatorial no sistema J2000 (unidade: graus)   |
|MAG_AUTO_{G,R,I,Z,Y}_DERED | Medida de magnitude aparente corrigida do avermelhamento da Galáxia (adimensional) |
|MAGERR_AUTO_{G,R,I,Z,Y}    | Incerteza na medida de magnitude aparente (adimensional) |
|z | Medida de _redshift_ espectroscópico |
|survey | Nome do levantamento espectroscópico que realizou a medida de _redshift_ |



### Acesso aos dados

Para simplificar a tarefa de aquisição dos dados &mdash; que envolveria o login na plataforma, a compreensão do método de cross-matching e a construção de uma query SQL para obter os dados &mdash; este repositório já contém um arquivo em formato `parquet` com os dados pré-selecionados.

In [ ]:
# Replace with wget command later
filepath = "./data/des_dr2/public_pz_training_set.pq" # O arquivo está no formato 'parquet'/
df = pd.read_parquet(filepath)

## Exercício 1: Use funcionalidades básicas da biblioteca pandas para verificar o formato e o conteúdo da tabela de dados

In [ ]:
# ---- INSIRA SEU CÓDIGO AQUI
# ...
# ----

## Exercício 2: Faça um gráfico da distribuição dos dados no céu. Depois selecione somente os levantamentos `VVDS`, `VIPERS` e `GAMA` e investigue a distribuição de cada um deles

### 2.a) Change coordinates as needed

In [ ]:
# ---- INSIRA SEU CÓDIGO AQUI
# ...
# ----

### 2.b) Make the plot

In [ ]:
def make_mollweide_plot(ra, dec):
    # ---- INSIRA SEU CÓDIGO AQUI
    # ...
    # ----
    return None

In [ ]:
# ---- INSIRA SEU CÓDIGO AQUI
# ...
# ----

### 2.c) Refaça a figura somente com os levantamentos `VVDS`, `VIPERS` e `GAMA`

In [ ]:
# ---- INSIRA SEU CÓDIGO AQUI
# ...
# ----

## Exercício 3: Analise as distribuições das galáxias nas variáveis pertinentes

### 3.a) Faça uma limpeza básica dos dados

In [ ]:
# ---- INSIRA SEU CÓDIGO AQUI
# ...
# ----

### 3.b) Investigue o número de galáxias por faixa de magnitude na banda $i$, total e para cada um dos levantamentos

In [ ]:
# ---- INSIRA SEU CÓDIGO AQUI
# ...
# ----

### 3.c) Investigue o número de galáxias por faixa de redshift, total e para cada um dos levantamentos

In [ ]:
# ---- INSIRA SEU CÓDIGO AQUI
# ...
# ----

### 3.d) Investigue a relação SNR vs mag_i separadamente para cada um dos levantamentos

In [ ]:
# ---- INSIRA SEU CÓDIGO AQUI
# ...
# ----

## Exercício 4: Crie novas colunas para as cores e investigue diagramas cor-cor, cor-redshift e mag_i-redshift

In [ ]:
# ---- INSIRA SEU CÓDIGO AQUI
# ...
# ----

In [ ]:
# ---- INSIRA SEU CÓDIGO AQUI
# ...
# ----

## Exercício 5: Baseado nas investigações dos exercícios 3 e 4, proponha cortes de qualidade 

In [ ]:
# ---- INSIRA SEU CÓDIGO AQUI
# ...
# ----

## Save data for photo-z measurements

In [ ]:
spec_sample_clean.to_parquet("./data/des_dr2/des_dr2_pz_training_set_clean_vvds_vipers_gama.pq", index=False)